In [2]:
# ── Excepción personalizada ──────────────────────────────────────────────────

class AnomaliaLanzamientoError(Exception):
    pass


# ── Decorador ────────────────────────────────────────────────────────────────

def auditar_protocol(func):
    def wrapper(*args, **kwargs):
        try:
            print("[SISTEMA] Iniciando protocolo de auditoría...")
            result = func(*args, **kwargs)
            print("[SISTEMA] Registro guardado en caja negra.")
            return result
        except AnomaliaLanzamientoError as e:
            print(f"🔴 ALERTA ROJA: {e}")
    return wrapper


# ── Clase abstracta ──────────────────────────────────────────────────────────

from abc import ABC, abstractmethod

class VehiculoEspacial(ABC):
    @abstractmethod
    def realizar_diagnostico(self):
        pass

    @abstractmethod
    def iniciar_secuencia(self):
        pass


# ── Clase concreta ───────────────────────────────────────────────────────────

import random

class CoheteFalcon(VehiculoEspacial):
    cohetes_activos = 0

    def __init__(self, nombre_mision):
        self.nombre_mision = nombre_mision
        self.__presion_motor = 0
        self.__listo_para_lanzamiento = False
        CoheteFalcon.cohetes_activos += 1

    @property
    def presion_motor(self):
        return self.__presion_motor

    @presion_motor.setter
    def presion_motor(self, valor):
        if 0 <= valor <= 100:
            self.__presion_motor = valor
        else:
            raise AnomaliaLanzamientoError("Presión del motor fuera de rango (0-100).")

    @classmethod
    def obtener_cohetes_activos(cls):
        return cls.cohetes_activos

    @auditar_protocol
    def realizar_diagnostico(self):
        self.presion_motor = random.randint(50, 110)
        if self.presion_motor > 100:
            raise AnomaliaLanzamientoError("Presión del motor excede el límite permitido.")
        self.__listo_para_lanzamiento = True
        print("Cohete Falcon: Diagnóstico completo, todos los sistemas operativos.")

    def iniciar_secuencia(self):
        if not self.__listo_para_lanzamiento:
            raise AnomaliaLanzamientoError("Cohete no listo para lanzamiento.")
        return f"¡Despegue nominal de la misión {self.nombre_mision}!"


# ── Prueba de cohetes ─────────────────────────────────────────────────────────

cohete1 = CoheteFalcon("Misión Alpha")
cohete2 = CoheteFalcon("Misión Beta")

cohete1.realizar_diagnostico()
print(cohete1.iniciar_secuencia())
print(cohete2.obtener_cohetes_activos())


# ── PokeAPI: generador de lotes ───────────────────────────────────────────────

def generar_lotes_pokemon(rango_inicio, rango_fin):
    for pokemon_id in range(rango_inicio, rango_fin + 1):
        url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}"
        yield url


# ── PokeAPI: extracción de métricas ──────────────────────────────────────────

import requests

def extraer_metricas(url):
    try:
        response = requests.get(url, timeout=3)
        response.raise_for_status()
        response_json = response.json()
        return {
            "name": response_json["name"],
            "height": response_json["height"],
            "weight": response_json["weight"]
        }
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            return None
        elif e.response.status_code == 500:
            print(f"Error de servidor (500) en {url}")
    except requests.exceptions.RequestException as e:
        print(f"Error de conexión en {url}: {e}")
    return None


# ── PokeAPI: concurrencia con Queue, Producers y Consumers ───────────────────

import queue
import threading

cola_de_tareas = queue.Queue()
base_datos_pokemon = []
candado_bd = threading.Lock()


def productor():
    print("Productor: Generando URLs de Pokemon...")
    for url in generar_lotes_pokemon(1, 50):
        cola_de_tareas.put(url)


def consumidor():
    while True:
        url = cola_de_tareas.get()
        if url is None:
            cola_de_tareas.task_done()
            break
        datos_pokemon = extraer_metricas(url)
        if datos_pokemon is not None:
            with candado_bd:
                base_datos_pokemon.append(datos_pokemon)
        cola_de_tareas.task_done()


consumidores = []
for i in range(5):
    t = threading.Thread(target=consumidor, daemon=True)
    t.start()
    consumidores.append(t)

productor_thread = threading.Thread(target=productor)
productor_thread.start()
productor_thread.join()

cola_de_tareas.join()

for _ in range(5):
    cola_de_tareas.put(None)

for t in consumidores:
    t.join()

print(f"Pokemons extraídos: {len(base_datos_pokemon)}")

[SISTEMA] Iniciando protocolo de auditoría...
Cohete Falcon: Diagnóstico completo, todos los sistemas operativos.
[SISTEMA] Registro guardado en caja negra.
¡Despegue nominal de la misión Misión Alpha!
2
Productor: Generando URLs de Pokemon...
Pokemons extraídos: 50
